# Jamaica connectivity mapping based on condition

### Step 1: Imports and set up base paths and output path

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from rasterio.warp import reproject
from rasterio.enums import Resampling
import connectivity
import tifffile
import rioxarray as rxr
import os
from osgeo import gdal
import rioxarray
import Robynlibrary as Robyn
import Robyn_forest_classes
from rasterio.windows import from_bounds
import matplotlib.pyplot as plt

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import rasterio, numpy as np
import geopandas as gpd
from pathlib import Path
import matplotlib.colors as mcolors


#### Define base paths and set inputs

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
input_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Inputs")

processed_dir = base_path / "Processed_data"
output_dir = base_path / "Outputs"

land_use_path = base_path / "Inputs/2013_landuse_LandCover.shp"
hydrobasins_path = processed_dir / "HydroBASINS_Level12_Clipped_Jamaica.shp"
output_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data")

jamaica_boundary_path = input_path / "Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

#### Read & reproject vector data

In [ ]:
terrestrial_landcover = gpd.read_file(land_use_path).copy() # Optional: if you want to preserve the original
jamaica_metric_grid_crs = "EPSG:3448"
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print("Reprojected CRS (terrestrial_landcover:", terrestrial_landcover.crs)

In [ ]:
hydrobasins = gpd.read_file(hydrobasins_path).copy()
print("Hydrobasins CRS:", hydrobasins.crs)

### Step 2: Read in condition spreadsheets and merge them with Jamaica landcover file 

In [ ]:
# Read your condition mapping spreadsheet
# mapping_table = pd.read_excel(base_path / "landcover_condition_mapping.xlsx")
test_condition_mapping_ones = pd.read_excel(base_path / "jamaica_landcover_condition_test_ones.xlsx")
test_condition_mapping_zeros = pd.read_excel(base_path / "jamaica_landcover_condition_test_zeros.xlsx")
# afforestable_condition_codes = pd.read_excel(base_path / "jamaica_afforestable_condition.xlsx")
baseline_condition_mixed_landcover_75_25_power4 = pd.read_excel(base_path / "landcover_condition_mapping_power4_75_25_mixed.xlsx")
baseline_condition_mixed_landcover_50_50_power4 = pd.read_excel(base_path / "landcover_condition_mapping_power4_50_50_mixed.xlsx")
afforestable_condition_power4_mixed_landcover_75_25_power4 = pd.read_excel(base_path / "jamaica_afforestable_condition_75_25_mixed_power4.xlsx")
afforestable_condition_power4_mixed_landcover_50_50_power4 = pd.read_excel(base_path / "jamaica_afforestable_condition_50_50_mixed_power4.xlsx")

In [ ]:
# Merge the mapping with your geodataframe on the 'Classify' column
# landcover_with_condition_baseline = terrestrial_landcover.merge(mapping_table, on='Classify')
landcover_with_condition_ones = terrestrial_landcover.merge(test_condition_mapping_ones, on='Classify')
landcover_with_condition_zeros = terrestrial_landcover.merge(test_condition_mapping_zeros, on='Classify')
# landcover_with_condition_afforestable = terrestrial_landcover.merge(afforestable_condition_codes, on='Classify')
landcover_with_condition_baseline_condition_mixed_landcover_75_25_power4 = terrestrial_landcover.merge(baseline_condition_mixed_landcover_75_25_power4, on='Classify')
landcover_with_condition_baseline_condition_mixed_landcover_50_50_power4 = terrestrial_landcover.merge(baseline_condition_mixed_landcover_50_50_power4, on='Classify')
landcover_with_condition_afforestable_condition_power4_mixed_landcover_75_25_power4 = terrestrial_landcover.merge(afforestable_condition_power4_mixed_landcover_75_25_power4, on='Classify')
landcover_with_condition_afforestable_condition_power4_mixed_landcover_50_50_power4 = terrestrial_landcover.merge(afforestable_condition_power4_mixed_landcover_50_50_power4, on='Classify')

### Step 3: Convert land use condition files to rasters

#### Define output files paths for the condition rasters 

In [ ]:
# Use the function for each condition and capture the outputs
# baseline_out = output_dir / "landcover_condition.tif"
ones_out = output_dir / "landcover_condition_test_ones.tif"
zeros_out = output_dir / "landcover_condition_test_zeros.tif"
# afforestable_out = output_dir / "landcover_condition_afforestable.tif"
baseline_condition_mixed_landcover_75_25_power4_out = output_dir / "baseline_landcover_condition_mixed_landcover_75_25_power4.tif"
baseline_condition_mixed_landcover_50_50_power4_out = output_dir / "baseline_landcover_condition_mixed_landcover_50_50_power4.tif"
afforestable_condition_power4_mixed_landcover_75_25_power4_out = output_dir / "afforestable_landcover_condition_75_25_power4.tif"
afforestable_condition_power4_mixed_landcover_50_50_power4_out = output_dir / "afforestable_landcover_condition_50_50_power4.tif"

#### Rasterize using the Robyn.rasterize_condition function 

In [ ]:
# condition_raster_baseline, transform = Robyn.rasterize_condition(landcover_with_condition_baseline, baseline_out)
# condition_raster_ones, _ = Robyn.rasterize_condition(landcover_with_condition_ones, ones_out)

condition_raster_ones, transform = Robyn.rasterize_condition(
    landcover_with_condition_ones,
    ones_out
)

condition_raster_zeros, _ = Robyn.rasterize_condition(landcover_with_condition_zeros, zeros_out)
# condition_raster_afforestable, _ = Robyn.rasterize_condition(landcover_with_condition_afforestable, afforestable_out)
condition_raster_baseline_condition_mixed_landcover_75_25_power4, _ = Robyn.rasterize_condition(
    landcover_with_condition_baseline_condition_mixed_landcover_75_25_power4,
    baseline_condition_mixed_landcover_75_25_power4_out
)
condition_raster_baseline_condition_mixed_landcover_50_50_power4, _ = Robyn.rasterize_condition(
    landcover_with_condition_baseline_condition_mixed_landcover_50_50_power4,
    baseline_condition_mixed_landcover_50_50_power4_out
)
afforestable_condition_raster_power4_mixed_landcover_75_25_power4, _ = Robyn.rasterize_condition(
    landcover_with_condition_afforestable_condition_power4_mixed_landcover_75_25_power4,
    afforestable_condition_power4_mixed_landcover_75_25_power4_out
)
afforestable_condition_raster_power4_mixed_landcover_50_50_power4, _ = Robyn.rasterize_condition(
    landcover_with_condition_afforestable_condition_power4_mixed_landcover_50_50_power4,
    afforestable_condition_power4_mixed_landcover_50_50_power4_out
)

### Step 4: resample to 100 by 100 size cells rather than the current 10 by 10 

In [ ]:
# 1) Compute the common spatial extent from your baseline GeoDataFrame
bounds = landcover_with_condition_ones.total_bounds

# 2) Define output file names for each resampled raster
ones_resampled_file = output_dir / "landcover_condition_test_ones_resampled.tif"
zeros_resampled_file = output_dir / "landcover_condition_test_zeros_resampled.tif"
baseline_75_25_power4_resampled_file = (
    output_dir / "baseline_landcover_condition_mixed_landcover_75_25_power4_resampled.tif"
)
baseline_50_50_power4_resampled_file = (
    output_dir / "baseline_landcover_condition_mixed_landcover_50_50_power4_resampled.tif"
)
afforestable_75_25_power4_resampled_file = (
    output_dir / "afforestable_landcover_condition_75_25_power4_resampled.tif"
)
afforestable_50_50_power4_resampled_file = (
    output_dir / "afforestable_landcover_condition_50_50_power4_resampled.tif"
)

# # 3) Resample each raster into that common extent & CRS
# Robyn.resample_and_save(
#     condition_raster_baseline,
#     transform,
#     landcover_with_condition_baseline.crs,
#     bounds,
#     baseline_resampled_file
# )

Robyn.resample_and_save(
    condition_raster_ones,
    transform,
    landcover_with_condition_ones.crs,
    bounds,
    ones_resampled_file
)

Robyn.resample_and_save(
    condition_raster_zeros,
    transform,
    landcover_with_condition_zeros.crs,
    bounds,
    zeros_resampled_file
)

Robyn.resample_and_save(
    condition_raster_baseline_condition_mixed_landcover_75_25_power4,
    transform,
    landcover_with_condition_baseline_condition_mixed_landcover_75_25_power4.crs,
    bounds,
    baseline_75_25_power4_resampled_file
)

Robyn.resample_and_save(
    condition_raster_baseline_condition_mixed_landcover_50_50_power4,
    transform,
    landcover_with_condition_baseline_condition_mixed_landcover_50_50_power4.crs,
    bounds,
    baseline_50_50_power4_resampled_file
)

Robyn.resample_and_save(
    afforestable_condition_raster_power4_mixed_landcover_75_25_power4,
    transform,
    landcover_with_condition_afforestable_condition_power4_mixed_landcover_75_25_power4.crs,
    bounds,
    afforestable_75_25_power4_resampled_file
)

Robyn.resample_and_save(
    afforestable_condition_raster_power4_mixed_landcover_50_50_power4,
    transform,
    landcover_with_condition_afforestable_condition_power4_mixed_landcover_50_50_power4.crs,
    bounds,
    afforestable_50_50_power4_resampled_file
)

### Step 5: Connectivity analysis 

In [ ]:
# # Compute connectivity for each resampled condition raster
# # baseline_connectivity = Robyn.compute_connectivity(baseline_resampled_file)
# ones_connectivity = Robyn.compute_connectivity(ones_resampled_file)
# zeros_connectivity = Robyn.compute_connectivity(zeros_resampled_file)
# baseline_75_25_power4_connectivity = Robyn.compute_connectivity(baseline_75_25_power4_resampled_file)
# baseline_50_50_power4_connectivity = Robyn.compute_connectivity(baseline_50_50_power4_resampled_file)
# afforestable_75_25_power4_connectivity = Robyn.compute_connectivity(afforestable_75_25_power4_resampled_file)
# afforestable_50_50_power4_connectivity = Robyn.compute_connectivity(afforestable_50_50_power4_resampled_file)

# # Print out the results
# # print(f"Baseline connectivity: {baseline_connectivity}")
# print(f"Test ones connectivity: {ones_connectivity}")
# print(f"Test zeros connectivity: {zeros_connectivity}")
# print(f"Baseline 75/25 power4 connectivity: {baseline_75_25_power4_connectivity}")
# print(f"Baseline 50/50 power4 connectivity: {baseline_50_50_power4_connectivity}")
# print(f"Afforestable 75/25 power4 connectivity: {afforestable_75_25_power4_connectivity}")
# print(f"Afforestable 50/50 power4 connectivity: {afforestable_50_50_power4_connectivity}")

In [ ]:
# # Compute connectivity using the already-defined resampled-file paths:
# # baseline_connectivity = Robyn.compute_connectivity(baseline_resampled_file)
ones_connectivity     = Robyn.compute_connectivity(ones_resampled_file)
zeros_connectivity    = Robyn.compute_connectivity(zeros_resampled_file)
baseline_75_25_conn   = Robyn.compute_connectivity(baseline_75_25_power4_resampled_file)
baseline_50_50_conn   = Robyn.compute_connectivity(baseline_50_50_power4_resampled_file)
afforest_75_25_conn   = Robyn.compute_connectivity(afforestable_75_25_power4_resampled_file)
afforest_50_50_conn   = Robyn.compute_connectivity(aafforestable_50_50_power4_resampled_file)

# Print out the results
# print(f"Baseline connectivity: {baseline_connectivity}")
print(f"Ones connectivity: {ones_connectivity}")
print(f"Zeros connectivity: {zeros_connectivity}")
print(f"Baseline 75/25 power4 connectivity: {baseline_75_25_conn}")
print(f"Baseline 50/50 power4 connectivity: {baseline_50_50_conn}")
print(f"Afforestable 75/25 power4 connectivity: {afforest_75_25_conn}")
print(f"Afforestable 50/50 power4 connectivity: {afforest_50_50_conn}")

In [ ]:
# Compute normalized connectivity (compared to the ones/zeros extremes) for each scenario
# baseline_normalized_extremes       = Robyn.calc_connectivity_normalized(
#     baseline_connectivity,
#     ones_connectivity,
#     zeros_connectivity
# )
baseline_75_25_normalized_extremes = Robyn.calc_connectivity_normalized(
    baseline_75_25_conn,
    ones_connectivity,
    zeros_connectivity
)
baseline_50_50_normalized_extremes = Robyn.calc_connectivity_normalized(
    baseline_50_50_conn,
    ones_connectivity,
    zeros_connectivity
)
afforest_75_25_normalized_extremes = Robyn.calc_connectivity_normalized(
    afforest_75_25_conn,
    ones_connectivity,
    zeros_connectivity
)
afforest_50_50_normalized_extremes = Robyn.calc_connectivity_normalized(
    afforest_50_50_conn,
    ones_connectivity,
    zeros_connectivity
)

# Print them out
# print(f"Baseline normalized (compared to extremes):                       {baseline_normalized_extremes:.2f}%")
print(f"Baseline 75/25 power4 normalized (compared to extremes):         {baseline_75_25_normalized_extremes:.2f}%")
print(f"Baseline 50/50 power4 normalized (compared to extremes):         {baseline_50_50_normalized_extremes:.2f}%")
print(f"Afforestable 75/25 power4 normalized (compared to extremes):     {afforest_75_25_normalized_extremes:.2f}%")
print(f"Afforestable 50/50 power4 normalized (compared to extremes):     {afforest_50_50_normalized_extremes:.2f}%")

In [ ]:
# Example values

# baseline_normalized_extremes = Robyn.calc_connectivity_normalized(baseline_connectivity, test_ones_connectivity, test_zeros_connectivity)
# afforestable_normalized_extremes = Robyn.calc_connectivity_normalized(afforestable_connectivity, test_ones_connectivity, test_zeros_connectivity)
# baseline_normalized_extremes_power4 = Robyn.calc_connectivity_normalized(baseline_connectivity_power4, ones_connectivity, zeros_connectivity)
# afforestable_normalized_extremes_power4 = Robyn.calc_connectivity_normalized(afforestable_connectivity_power4, ones_connectivity, zeros_connectivity)

# # print(f"Baseline connectivity normalized (compared to extremes): {baseline_normalized_extremes:.2f}%")
# # print(f"Afforestable connectivity normalized (compared to extremes): {afforestable_normalized_extremes:.2f}%")
# print(f"Baseline connectivity power 4 normalized (compared to extremes): {baseline_normalized_extremes_power4:.2f}%")
# print(f"Afforestable connectivity power 4 normalized (compared to extremes): {afforestable_normalized_extremes_power4:.2f}%")

# # Calculate the improvement (difference in percentage points)
# # improvement = afforestable_normalized_extremes - baseline_normalized_extremes
# improvement_power4 = afforestable_normalized_extremes_power4 - baseline_normalized_extremes_power4
# # print(f"Improvement in connectivity percentage from baseline to afforestable: {improvement:.2f}%")
# print(f"Improvement in connectivity percentage from baseline to afforestable power4: {improvement_power4:.2f}%")

# # relative_improvement = ((afforestable_normalized_extremes - baseline_normalized_extremes) / baseline_normalized_extremes) * 100
# relative_improvement_power4 = ((afforestable_normalized_extremes_power4 - baseline_normalized_extremes_power4) / baseline_normalized_extremes_power4) * 100
# # print(f"Relative improvement compared to baseline: {relative_improvement:.2f}%")
# print(f"Relative improvement compared to baseline power4: {relative_improvement_power4:.2f}%")


### Step 6: calculate forest area as a proportion of Jamaica land cover area 

In [ ]:
# 1) total land area
total_land_area = terrestrial_landcover.geometry.area.sum()

# 2) your two class‐sets
flood_classes = Robyn_forest_classes.forest_flood_equivalent_classes
afforest_classes = Robyn_forest_classes.afforestable_classes_including_agricultural

# 4) build the union for the future scenario
future_classes = flood_classes.union(afforest_classes)

# 5) flatten your mixed‐fractions for the future scenario
mixed = Robyn_forest_classes.mixed_land_use_fractions
future_mixed = {
    cls: fracs['afforestable_including_agriculture']
    for cls, fracs in mixed.items()
    if 'afforestable_including_agriculture' in fracs
}

# 6) compute areas (m²)
baseline_area = Robyn.compute_forest_area(
    terrestrial_landcover,
    flood_classes,
    mixed={cls: fracs['forest_flood_equivalent_classes']
           for cls, fracs in mixed.items()
           if 'forest_flood_equivalent_classes' in fracs}
)

future_area   = Robyn.compute_forest_area(
    terrestrial_landcover,
    future_classes,
    mixed=future_mixed
)

# 7) percentages & km²
baseline_pct = baseline_area / total_land_area * 100
future_pct   = future_area   / total_land_area * 100

baseline_km2 = baseline_area / 1e6
future_km2   = future_area   / 1e6
total_km2    = total_land_area / 1e6

print(f"Total land: {total_land_area:,.0f} m² ({total_km2:.2f} km²)")
print(f"Baseline forest: {baseline_area:,.0f} m² ({baseline_km2:.2f} km²) → {baseline_pct:.2f}%")
print(f"Future forest:   {future_area:,.0f} m² ({future_km2:.2f} km²) → {future_pct:.2f}%")

### Step 7: Catchment-level analysis 

In [ ]:
# Calculate the area in m² and km², and add them as new columns
hydrobasins["area_m2"] = hydrobasins.geometry.area
hydrobasins["area_km2"] = hydrobasins["area_m2"] / 1e6

# Add a new column with a unique new ID starting at 1
hydrobasins["new_id"] = range(1, len(hydrobasins) + 1)

# Calculate the equivalent diameter (distance across) in meters
# Equivalent diameter = 2 * sqrt(area_m2 / pi)
hydrobasins["equiv_diam_m"] = 2 * np.sqrt(hydrobasins["area_m2"] / np.pi)


# Calculate the smallest and largest HYBAS_ID area (in m² and km²)
smallest_area_m2 = hydrobasins["area_m2"].min()
mean_area_m2 = hydrobasins["area_m2"].mean()
largest_area_m2 = hydrobasins["area_m2"].max()
smallest_area_km2 = hydrobasins["area_km2"].min()
mean_area_km2 = hydrobasins["area_km2"].mean()
largest_area_km2 = hydrobasins["area_km2"].max()

smallest_diam = hydrobasins["equiv_diam_m"].min()
mean_diam = hydrobasins["equiv_diam_m"].mean()
largest_diam = hydrobasins["equiv_diam_m"].max()


# Print summary statistics of hydrobasins

print("Smallest HYBAS_ID area (m²):", smallest_area_m2)
print("Mean HYBAS_ID area (m²):", mean_area_m2)
print("Largest HYBAS_ID area (m²):", largest_area_m2)
print("Smallest HYBAS_ID area (km²):", smallest_area_km2)
print("Mean HYBAS_ID area (km²):", mean_area_km2)
print("Largest HYBAS_ID area (km²):", largest_area_km2)

print("Smallest equivalent diameter (m):", smallest_diam)
print("Mean equivalent diameter (m):", mean_diam)
print("Largest equivalent diameter (m):", largest_diam)
number_catchments = hydrobasins["new_id"].max()
print(number_catchments)

# Optionally, print a preview of the new columns along with the HYBAS_ID
print(hydrobasins[['HYBAS_ID', 'new_id', 'area_m2', 'area_km2', 'equiv_diam_m']].head())

# Write the modified hydrobasins to a new shapefile
hydrobasins.to_file("HydroBASINS_Level12_Clipped_Jamaica_modified.shp")

#### Rasterize hydrobasins 

In [ ]:
# Define the raster resolution (in meters) and extent
pixel_size = 100  # Change this value to your desired resolution (e.g., 10m)
minx, miny, maxx, maxy = hydrobasins.total_bounds

# Compute width and height in terms of pixels
width = int(np.ceil((maxx - minx) / pixel_size))
height = int(np.ceil((maxy - miny) / pixel_size))

# Create an affine transform for the raster (origin at top-left)
transform = from_origin(minx, maxy, pixel_size, pixel_size)

# Create (geometry, value) pairs for rasterization using the "new_id" column as the value.
shapes = ((geom, value) for geom, value in zip(hydrobasins.geometry, hydrobasins['new_id']))

# Rasterize the geometries into a NumPy array. 
raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,  # Pixels that don't fall within any geometry will be assigned the "fill" value (0).
    dtype=np.uint16  # Change type as needed based on your data range
)

In [ ]:
# Define the output file path for saving the hydrobasins raster to a GeoTIFF file
hydrobasins_raster_path = processed_dir / "hydrobasins_raster.tif"
clipped_hydro_raster    = output_dir    / "hydrobasins_clipped.tif"

# print("Reprojected raster saved to:", output_dir / "hydrobasins_raster_reproj.tif")
baseline_raster_path = output_dir / "landcover_condition_resampled.tif"
with rasterio.open(str(hydrobasins_raster_path)) as src, rasterio.open(str(baseline_raster_path)) as tmpl:
    # get the exact bounds & transform of your template
    bounds    = tmpl.bounds          # (left, bottom, right, top)
    transform = src.transform

    # build a window in the source that matches those bounds
    window    = from_bounds(*bounds, transform=transform)

    # read only that window from band 1
    data      = src.read(1, window=window)
    # compute the transform for that window
    new_transform = src.window_transform(window)

    # update the profile for writing
    profile = src.profile.copy()
    profile.update({
        "height": window.height,
        "width":  window.width,
        "transform": new_transform
    })

    # write out the clipped, aligned raster
    with rasterio.open(str(clipped_hydro_raster), "w", **profile) as dst:
        dst.write(data, 1)

print("Clipped hydrobasins raster saved to:", clipped_hydro_raster)

In [ ]:
with rasterio.open(clipped_hydro_raster) as src:
    transform = src.transform
    bounds    = src.bounds
    width, height = src.width, src.height
    print("Clipped hydrobasins raster:")
    print("  CRS:",             src.crs)
    print("  Width × Height:",  width, "×", height)
    print("  Pixel size:",      transform.a, "×", transform.e)
    print("  Bounds:",          bounds)

with rasterio.open(baseline_power4_resampled_file) as tmpl: 
    print("Template landcover raster:")
    print("  CRS:", tmpl.crs)
    print("  Width × Height:", tmpl.width, "×", tmpl.height)
    print("  Pixel size:",     tmpl.transform.a, "×", tmpl.transform.e)
    print("  Bounds:",         tmpl.bounds)


In [ ]:
catchment_raster_upd          = Robyn.open_raster_as_array(str(clipped_hydro_raster))
baseline_condition_raster_power4_upd = Robyn.open_raster_as_array(str(baseline_power4_resampled_file))
afforested_condition_raster_power4_upd = Robyn.open_raster_as_array(str(afforestable_power4_resampled_file))

print("Unique catchment IDs:", np.unique(catchment_raster_upd))
print("Catchment raster shape:",  catchment_raster_upd.shape)
print("Baseline raster shape:",   baseline_condition_raster_power4_upd.shape)
print("Afforested raster shape:",   afforested_condition_raster_power4_upd.shape)



In [ ]:
number_catchments = int(catchment_raster_upd.max())  # should match hydrobasins["new_id"].max()

# 1) pixel counts
for cid in range(1, number_catchments+1):
    mask = (catchment_raster_upd == cid)
    pixel_count = np.count_nonzero(mask)
    print(f"Catchment {cid}: non-zero pixels = {pixel_count}")

# 2) connectivity
n_processes                 = os.cpu_count()
#lambda_parameter            = 10
generations_mode            = "one_generation"
number_of_species_generations = 1
zeros_resampled_array = Robyn.open_raster_as_array(zeros_resampled_file)
ones_resampled_array = Robyn.open_raster_as_array(ones_resampled_file)

for lambda_parameter in [0.5, 5]: #1, 10, with a resolution of 100m this would translate into distances of 100m, 1km, 10km as it is pixel size
    for cid in range(1, number_catchments+1):
        mask = (catchment_raster_upd == cid).astype(np.uint8)
    
        #calculate baseline connectivity for catchment
        base_conn = connectivity.landscape_connectivity(
            baseline_condition_raster_power4_upd,
            n_processes,
            mask,
            lambda_parameter,
            generations_mode,
            number_of_species_generations)
        
        #calculate afforested connectivity for catchment
        aff_conn = connectivity.landscape_connectivity(
            afforested_condition_raster_power4_upd,
            n_processes,
            mask,
            lambda_parameter,
            generations_mode,
            number_of_species_generations)
        
        #calculate "test_zeros" connectivity for catchement
        test_zeros_conn_catchment = connectivity.landscape_connectivity(
            zeros_resampled_array,
            n_processes,
            mask,
            lambda_parameter,
            generations_mode,
            number_of_species_generations)
    
        #calculate "test_ones" connectivity for catchement
        test_ones_conn_catchment = connectivity.landscape_connectivity(
            ones_resampled_array,
            n_processes,
            mask,
            lambda_parameter,
            generations_mode,
            number_of_species_generations)
    
    
        #calculate "baseline_power4" connectivity for catchement
        baseline_conn_power4 = connectivity.landscape_connectivity(
            baseline_condition_raster_power4_upd,
            n_processes,
            mask,
            lambda_parameter,
            generations_mode,
            number_of_species_generations)
    
        #calculate afforested connectivity for catchment
        aff_conn_power4 = connectivity.landscape_connectivity(
            afforested_condition_raster_power4_upd,
            n_processes,
            mask,
            lambda_parameter,
            generations_mode,
            number_of_species_generations)
    
    
        
        #calculate percentage improvement compared to baseline connectivity
        perc_diff = (aff_conn - base_conn) / base_conn * 100
        perc_diff_power4 = (aff_conn_power4 - baseline_conn_power4) / baseline_conn_power4 * 100
    
        
        baseline_normalized = Robyn.calc_connectivity_normalized(base_conn, test_ones_conn_catchment, test_zeros_conn_catchment)
        aff_normalized = Robyn.calc_connectivity_normalized(aff_conn, test_ones_conn_catchment, test_zeros_conn_catchment)
        print(f"Connectivity catchment {cid}: "
              f"baseline={base_conn:.2f}, baseline normalized={baseline_normalized:.3f}, afforested={aff_conn:.2f}, "
              f"diff={perc_diff:.1f}%, afforested normalized={aff_normalized:.3f}")
    
        baseline_normalized_power4 = Robyn.calc_connectivity_normalized(baseline_conn_power4, test_ones_conn_catchment, test_zeros_conn_catchment)
        aff_normalized_power4 = Robyn.calc_connectivity_normalized(aff_conn_power4, test_ones_conn_catchment, test_zeros_conn_catchment)
        print(f"Connectivity catchment {cid} power4: "
              f"baseline_power4={baseline_conn_power4:.2f}, baseline normalized power 4={baseline_normalized_power4:.3f}, afforested={aff_conn_power4:.2f}, "
              f"diff_power4={perc_diff_power4:.1f}%, afforested normalized power 4={aff_normalized_power4:.3f}")
    




        # if you want to store back into your GeoDataFrame:
        hydrobasins.loc[hydrobasins["new_id"] == cid, [
            "baseline_catchment_connectivity",
            "baseline_normalized",
            "afforested_catchment_connectivity",
            "percentage_difference_catchment_connectivity",
            "afforested_normalized_extremes",
            "baseline_catchment_connectivity_power4",
            "baseline_normalized_power4",
            "afforested_catchment_connectivity_power4",
            "percentage_difference_catchment_connectivity_power4",
            "afforested_normalized_power4",
        ]] = [base_conn, baseline_normalized, aff_conn, perc_diff, aff_normalized, baseline_conn_power4, baseline_normalized_power4, aff_conn_power4, perc_diff_power4, aff_normalized_power4 ]
    
    #save table as excel file
    keep_cols = [
        "HYBAS_ID",
        "area_m2",
        "area_km2",
        "new_id",
        "equiv_diam_m",
        "baseline_catchment_connectivity",
        "baseline_normalized",
        "afforested_catchment_connectivity",
        "percentage_difference_catchment_connectivity",
        "afforested_normalized_extremes",
        "baseline_catchment_connectivity_power4",
        "baseline_normalized_power4",
        "afforested_catchment_connectivity_power4",
        "percentage_difference_catchment_connectivity_power4",
        "afforested_normalized_power4",
    ]
    
    # Subset and export
    lambda_out = hydrobasins[keep_cols]
    lambda_out.to_csv(f"{output_dir}/lambda_{lambda_parameter}_catchment_connectivity.csv", index=False)
    
    
    
    
    # hydrobasins.to_csv(f'{output_dir}/lambda_{lambda_parameter}_catchment_connectivity.csv', index=False)
    print(f"Done with lambda value {lambda_parameter}!")





In [ ]:
# # 1. Define your paths
# conn_csv   = Path(output_dir) / "catchment_connectivity.csv"
# forest_csv = Path(processed_dir) / "catchment_forest_summary_with_percentages.csv"

# # 2. Read both tables
# df_conn   = pd.read_csv(conn_csv)
# df_forest = pd.read_csv(forest_csv)

# # 3. Merge on HYBAS_ID (inner join keeps only IDs in both; use how="left" or "outer" if you need)
# df_merged = df_conn.merge(df_forest, on="HYBAS_ID", how="inner")

# # 4. (Optional) inspect
# display(df_merged.columns)
# display(df_merged.head())

# # 5. Save your combined table
# df_merged.to_csv(Path(output_dir) / "lambda_10_catchment_connectivity_forest_area_combined.csv", index=False)

# # Round all numeric columns to 2 decimals
# df_merged = df_merged.round(2)

# # Save
# df_merged.to_csv(
#     Path(output_dir) / "lambda_10_catchment_connectivity_forest_area_combined.csv",
#     index=False
# )

In [ ]:
original_hydrobasins = gpd.read_file(input_path / "Hydrobasins_12/hybas_na_lev12_v1c.shp")
reprojected_hydrobasins = original_hydrobasins.to_crs(jamaica_boundary.crs)

# 2) Clip to the Jamaica boundary
hydro = gpd.clip(reprojected_hydrobasins, jamaica_boundary)

# 3) Write out once—and keep the same object for further use
hydro.to_file(output_path / "HydroBASINS_Level12_Clipped_Jamaica.shp")
print("Clipped CRS:", hydro.crs)


In [ ]:
terrestrial_landcover_hydrobasins_intersection = gpd.overlay(
    terrestrial_landcover, hydro, how="intersection"
)


In [ ]:
conn = pd.read_csv(output_dir / "catchment_connectivity.csv", dtype={"HYBAS_ID": str})
conn['HYBAS_ID'] = conn['HYBAS_ID'].astype(int)

# merge and dissolve as before
hydro = hydro.merge(
    conn[['HYBAS_ID','baseline_normalized_power4','new_id']],
    on='HYBAS_ID', how='left'
)
hydro = hydro.dissolve(by='new_id', aggfunc='mean').reset_index()

# --- 2) Build color ramp on [0, max] of baseline_normalized_power4 ---
cmap = plt.colormaps['Greens']
vmax = hydro['baseline_normalized_power4'].max()
norm = mpl.colors.Normalize(vmin=0, vmax=vmax)

hydro['color'] = hydro['baseline_normalized_power4'] \
    .map(lambda val: mcolors.to_hex(cmap(norm(val))))

# --- 3) Scale bar & north arrow helpers ---
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), fontsize=14, lw=3):
    half = 0.05; x,y = location
    ax.plot([x-half, x+half], [y, y], transform=ax.transAxes, color='black', lw=2)
    for pos in (x-half, x, x+half):
        ax.plot([pos, pos], [y-0.005, y+0.005], transform=ax.transAxes, color='black', lw=2)
    ax.text(x-half, y-0.03, "0", transform=ax.transAxes, ha='center', va='center')
    ax.text(x,       y-0.03, f"{length_km//2}", transform=ax.transAxes, ha='center', va='center')
    ax.text(x+half,  y-0.03, f"{length_km}",    transform=ax.transAxes, ha='center', va='center')
    ax.text(x+half+0.02, y, "km", transform=ax.transAxes, ha='left', va='center')

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=14, lw=3):
    x,y = location
    ax.annotate(
        "",
        xy=(x, y+size), xycoords='axes fraction',
        xytext=(x, y),   textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black',
                        headwidth=10, headlength=15, width=5)
    )
    ax.text(x, y+size+0.02, "N", transform=ax.transAxes,
            ha='center', va='center', fontsize=12, fontweight='bold')

# --- 4) Plot & save function ---
def plot_baseline_connectivity(gdf, title, fname):
    fig, ax = plt.subplots(figsize=(18,14), dpi=300)

    # fill catchments
    gdf.plot(ax=ax, color=gdf['color'], linewidth=0, alpha=0.8)

    # boundaries
    gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)

    # continuous colorbar
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm._A = []
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label("Baseline normalized connectivity (power 4)", family='Times New Roman')

    # catchment labels
    for _, row in gdf.iterrows():
        pt = row.geometry.representative_point()
        ax.text(pt.x, pt.y, str(int(row.new_id)),
                ha='center', va='center', fontsize=6, fontweight='bold')

    # legend stub for boundaries
    handle = Line2D([0], [0], color='black', lw=1.5, label='Catchment boundary')
    leg = ax.legend(handles=[handle], title='Legend',
              bbox_to_anchor=(0.5, -0.15), loc='upper center',
                    frameon=False,
                    fontsize=20,
                    title_fontsize=16,
                    prop={'family':'Times New Roman'})
    
    leg.get_title().set_position((0, 10))
    # add cartographic elements
    add_scale_bar(ax)
    add_north_arrow(ax)

    ax.set_axis_off()
    plt.title(title, fontsize=20, fontweight='bold',
              fontname='Times New Roman', pad=20)
    plt.tight_layout()

    # save to disk
    out_png = Path(output_dir)/f"{fname}.png"
    out_pdf = Path(output_dir)/f"{fname}.pdf"
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    fig.savefig(out_pdf,              bbox_inches='tight')
    print(f"Saved {out_png} and {out_pdf}")

    plt.show()

# --- 5) Call it ---
plot_baseline_connectivity(
    hydro,
    title="Baseline Normalized Connectivity (Power 4)\nby Catchment in Jamaica",
    fname="Figure_baseline_connectivity_power4"
)

In [ ]:
from Robyn_forest_classes import (
    forest_flood_equivalent_classes,
    afforestable_classes_including_agricultural as afforestable_classes,
    mixed_land_use_fractions
)

# 1) read your catchments + connectivity
# hydro = gpd.read_file(output_path / "HydroBASINS_Level12_Clipped_Jamaica.shp")
# conn = pd.read_csv(output_dir / "catchment_connectivity.csv", dtype={"HYBAS_ID":int})
# hydro = hydro.merge(
#    conn[['HYBAS_ID','baseline_normalized_power4','afforested_normalized_power4','new_id']],
#    on='HYBAS_ID', how='left'
#).dissolve(by='new_id', aggfunc='mean').reset_index()

# 2) read your land-use once
lu = gpd.read_file(input_path / "2013_landuse_LandCover.shp")

# 3) define your two fraction functions
def flood_frac(cls):
    if cls in mixed_land_use_fractions:
        return mixed_land_use_fractions[cls].get('forest_flood_equivalent_classes',0)
    return 1 if cls in forest_flood_equivalent_classes else 0

def future_frac(cls):
    if cls in mixed_land_use_fractions:
        return mixed_land_use_fractions[cls].get('afforestable_including_agriculture',0)
    return 1 if (cls in forest_flood_equivalent_classes or cls in afforestable_classes) else 0

# 4) generic scenario runner
def run_scenario(name, frac_func, hydro, value_col, out_fname):
    # compute fraction & filter
    lu[f"{name}_frac"] = lu['Classify'].map(frac_func).fillna(0.0)
    patches = lu[lu[f"{name}_frac"] > 0].to_crs(hydro.crs)

    # intersect & merge connectivity
    patches_int = gpd.overlay(patches, hydro[['new_id','geometry']], how='intersection')
    patches_int = patches_int.merge(hydro[['new_id', value_col]], on='new_id', how='left')

    # plot
    cmap = plt.colormaps['Greens']
    norm = mpl.colors.Normalize(vmin=0, vmax=hydro[value_col].max())
    patches_int['color'] = patches_int[value_col].map(lambda v: mcolors.to_hex(cmap(norm(v))))

    fig, ax = plt.subplots(figsize=(18,14), dpi=300)
    patches_int.plot(ax=ax, color=patches_int['color'], linewidth=0, alpha=0.8)
    hydro.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)

    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm._A=[]
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label(f"{name} normalized connectivity (power 4)", family='Times New Roman')

    for _, row in hydro.iterrows():
        pt = row.geometry.representative_point()
        ax.text(pt.x, pt.y, str(int(row.new_id)),
                ha='center', va='center', fontsize=6, fontweight='bold')

    handle = Line2D([0],[0], color='black', lw=0.5, label='Catchment boundary')
    ax.legend(handles=[handle], title='Legend',
              bbox_to_anchor=(0.5,-0.1), loc='upper center',
              frameon=False, fontsize=12, title_fontsize=14)

    # (re-use your scale/north arrow functions here)
    add_scale_bar(ax); add_north_arrow(ax)

    ax.set_axis_off()
    plt.title(f"{name.capitalize()} Connectivity on Forest Patches",
              fontsize=20, fontweight='bold', family='Times New Roman', pad=20)
    plt.tight_layout()

    fig.savefig(Path(output_dir)/f"{out_fname}.png", dpi=300, bbox_inches='tight')
    fig.savefig(Path(output_dir)/f"{out_fname}.pdf",              bbox_inches='tight')
    plt.show()


# 5) run both
# run_scenario(
#     name="baseline",
#     frac_func=flood_frac,
#     hydro=hydro,
#     value_col="baseline_normalized_power4",
#     out_fname="Figure_baseline_connectivity_on_forest"
# )

# run_scenario(
#     name="future",
#     frac_func=future_frac,
#     hydro=hydro,
#     value_col="afforested_normalized_power4",
#     out_fname="Figure_future_connectivity_on_forest"
# )
#########################################################################
for lambda_parameter in [1, 10, 100]:
    # 1) read your catchments + connectivity
    hydro = gpd.read_file(output_path / "HydroBASINS_Level12_Clipped_Jamaica.shp")
    conn = pd.read_csv(output_dir / f"lambda_{lambda_parameter}_catchment_connectivity.csv", dtype={"HYBAS_ID":int})
    hydro = hydro.merge(
        conn[['HYBAS_ID','baseline_normalized_power4','afforested_normalized_power4','new_id']],
        on='HYBAS_ID', how='left'
    ).dissolve(by='new_id', aggfunc='mean').reset_index()
    
    # 5) run both
    run_scenario(
        name="baseline",
        frac_func=flood_frac,
        hydro=hydro,
        value_col="baseline_normalized_power4",
        out_fname=f"Figure_baseline_connectivity_on_forest_lambda{lambda_parameter}"
    )
    
    run_scenario(
        name="future",
        frac_func=future_frac,
        hydro=hydro,
        value_col="afforested_normalized_power4",
        out_fname=f"Figure_future_connectivity_on_forest_lambda{lambda_parameter}"
    )

In [ ]:
# this CSV can be your first `catchment_connectivity.csv` that you produced once
mapping_df = pd.read_csv(Path(output_dir)/"catchment_connectivity.csv",
                         dtype={"HYBAS_ID":int})[['HYBAS_ID','new_id']]

In [ ]:

# --- your fraction funcs & plot helper must already be in scope: ---
# flood_frac, future_frac
# add_scale_bar, add_north_arrow
# run_scenario(name, frac_func, value_col, out_fname)

# where to write
output_dir = Path(output_dir)
shp_path   = output_path / "HydroBASINS_Level12_Clipped_Jamaica.shp"

# load static HYBAS_ID→new_id mapping once
mapping_df = pd.read_csv(
    output_dir/"catchment_connectivity.csv",
    dtype={"HYBAS_ID":int}
)[['HYBAS_ID','new_id']]

lambda_values = [5, 10, 50]
number_catchments = int(catchment_raster_upd.max())

for lam in lambda_values:
    print(f"\n⚙️  Running connectivity calc with λ = {lam}")

    # load & tag catchments
    hydro_gdf = gpd.read_file(shp_path)
    hydro_gdf = hydro_gdf.merge(mapping_df, on="HYBAS_ID", how="left")
    
    # will collect one row per catchment
    rows = []
    
    # pre‐load the zero/one rasters once per λ
    zeros = Robyn.open_raster_as_array(zeros_resampled_file)
    ones  = Robyn.open_raster_as_array(ones_resampled_file)
    
    for cid in range(1, number_catchments+1):
        mask = (catchment_raster_upd == cid).astype(np.uint8)
        
        # --- your connectivity calls ---
        base_conn = connectivity.landscape_connectivity(
            baseline_condition_raster_upd,
            os.cpu_count(), mask, lam,
            "one_generation", 1)
        
        aff_conn = connectivity.landscape_connectivity(
            afforested_condition_raster_upd,
            os.cpu_count(), mask, lam,
            "one_generation", 1)
        
        baseline_conn_power4 = connectivity.landscape_connectivity(
            baseline_condition_raster_power4_upd,
            os.cpu_count(), mask, lam,
            "one_generation", 1)
        
        aff_conn_power4 = connectivity.landscape_connectivity(
            afforested_condition_raster_power4_upd,
            os.cpu_count(), mask, lam,
            "one_generation", 1)
        # -------------------------------

        # % change
        pctchg_p4 = ((aff_conn_power4 - baseline_conn_power4)
                     / baseline_conn_power4 * 100
                     if baseline_conn_power4 != 0 else np.nan)

        # normalized
        base_norm_p4 = Robyn.calc_connectivity_normalized(
            baseline_conn_power4, ones, zeros)
        aff_norm_p4  = Robyn.calc_connectivity_normalized(
            aff_conn_power4,     ones, zeros)

        rows.append({
            "new_id": cid,
            "baseline_normalized_power4": base_norm_p4,
            "afforested_normalized_power4": aff_norm_p4,
            "percentage_difference_catchment_connectivity_power4": pctchg_p4
        })

    # build results DataFrame
    df_res = pd.DataFrame(rows)

    # merge into GeoDataFrame
    hydro = hydro_gdf.merge(df_res, on="new_id", how="left")

    # save CSV
    csv_path = output_dir/f"catchment_connectivity_lambda{lam}.csv"
    hydro[[
        "HYBAS_ID",
        "new_id",
        "baseline_normalized_power4",
        "afforested_normalized_power4",
        "percentage_difference_catchment_connectivity_power4"
    ]].to_csv(csv_path, index=False)
    print(f"→ wrote {csv_path.name}")

    # dissolve by new_id for mapping
    hydro_d = hydro.dissolve(
        by="new_id",
        aggfunc="mean"
    ).reset_index()

    # make the three maps
    run_scenario(
        name=f"baseline_λ{lam}",
        frac_func=flood_frac,
        value_col="baseline_normalized_power4",
        out_fname=f"Figure_baseline_conn_power4_lambda{lam}"
    )
    run_scenario(
        name=f"future_λ{lam}",
        frac_func=future_frac,
        value_col="afforested_normalized_power4",
        out_fname=f"Figure_future_conn_power4_lambda{lam}"
    )
    run_scenario(
        name=f"pctchange_λ{lam}",
        frac_func=lambda cls: 1,
        value_col="percentage_difference_catchment_connectivity_power4",
        out_fname=f"Figure_pctchange_conn_power4_lambda{lam}"
    )

print("✅ All λ runs complete.")

#### Load rasters as arrays for further processing / validation 

In [ ]:
# Load and verify the values from the original and reprojected hydrobasins rasters.
original_hydrobasins_array = Robyn.open_raster_as_array(str(hydrobasins_raster_path))
reprojected_hydrobasins_array = Robyn.open_raster_as_array(str(output_dir / "hydrobasins_raster_reproj.tif"))

print("Unique hydrobasin IDs in original raster:", np.unique(original_hydrobasins_array))
print("Unique hydrobasin IDs in reprojected raster:", np.unique(reprojected_hydrobasins_array))

print("Shape of the reprojected hydrobasins raster:", reprojected_hydrobasins_array.shape)
print("Shape of the landcover bounds:", baseline_array.shape)


In [ ]:

# # 1) Read mapping of HYBAS_ID → new_id
# mapping_df = pd.read_csv(
#     Path(output_dir)/"catchment_connectivity.csv",
#     dtype={"HYBAS_ID": str}
# )[['HYBAS_ID','new_id']]

In [ ]:
# Read the baseline condition raster.
baseline_condition_raster = Robyn.open_raster_as_array(str(baseline_resampled_file))
print("Shape of catchment raster:", np.shape(reprojected_hydrobasins_array))
print("Shape of baseline condition raster:", np.shape(baseline_condition_raster))

# Determine the total number of catchments from the hydrobasins GeoDataFrame.
number_catchments = hydrobasins["new_id"].max()

# Loop over each catchment and calculate the non-zero pixel count.
for catchment_number in range(1, number_catchments + 1):
    catchment_temp = np.zeros_like(baseline_condition_raster)
    catchment_temp[np.where(reprojected_hydrobasins_array == catchment_number)] = 1
    pixel_count = np.count_nonzero(catchment_temp)
    print(f"Catchment {catchment_number} - non-zero pixel count: {pixel_count}")

# Read the afforestable (afforested) condition raster.
afforested_condition_raster = Robyn.open_raster_as_array(str(afforestable_resampled_file))

# Setup parameters for connectivity analysis.
n_processes = os.cpu_count()
lambda_parameter = 5
generations_mode = "one_generation"
number_of_species_generations = 1

# Loop over each catchment to compute connectivity values.
for catchment_number in range(1, number_catchments + 1):
    # Create a binary mask for the current catchment.
    catchment_temp = np.zeros_like(baseline_condition_raster)
    catchment_temp[np.where(reprojected_hydrobasins_array == catchment_number)] = 1
             
    # Compute connectivity for the current catchment under baseline conditions.
    baseline_connectivity_value = connectivity.landscape_connectivity(
        baseline_condition_raster, 
        n_processes, 
        catchment_temp, 
        lambda_parameter, 
        generations_mode, 
        number_of_species_generations
    )
    # Compute connectivity for the current catchment under the afforested scenario.
    afforested_connectivity_value = connectivity.landscape_connectivity(
        afforested_condition_raster, 
        n_processes, 
        catchment_temp, 
        lambda_parameter, 
        generations_mode, 
        number_of_species_generations
    )

    # Calculate the percentage difference in connectivity.
    perc_difference = (((afforested_connectivity_value - baseline_connectivity_value) / baseline_connectivity_value) * 100)
    print(f"Connectivity catchment {catchment_number}: baseline = {baseline_connectivity_value}, afforested = {afforested_connectivity_value}, difference = {perc_difference:.2f}%")
    
    # Update the hydrobasins GeoDataFrame with these connectivity metrics.
    hydrobasins.loc[hydrobasins["new_id"] == catchment_number, "baseline_catchment_connectivity"] = baseline_connectivity_value
    hydrobasins.loc[hydrobasins["new_id"] == catchment_number, "afforested_catchment_connectivity"] = afforested_connectivity_value
    hydrobasins.loc[hydrobasins["new_id"] == catchment_number, "percentage_difference_catchment_connectivity"] = perc_difference


### *** OLD CODE (for applying to one file at a time) *** 

In [ ]:
# # Determine the bounds and resolution for the output raster
# minx, miny, maxx, maxy = landcover_with_condition_baseline.total_bounds
# resolution = 10  # Define an appropriate resolution (in the units of your CRS)
# width = int((maxx - minx) / resolution)
# height = int((maxy - miny) / resolution)
# transform = from_origin(minx, maxy, resolution, resolution)

In [ ]:
# # Prepare shapes for rasterization: tuple of (geometry, condition value)
# shapes = ((geom, value) for geom, value in zip(landcover_with_condition.geometry, landcover_with_condition['Condition']))


In [ ]:
# condition_raster = rasterize(
#     shapes=shapes,
#     out_shape=(height, width),
#     fill=0,  # Value for areas with no data
#     transform=transform,
#     dtype='float32'
# )

In [ ]:
# # Write the raster to a GeoTIFF
# with rasterio.open(
#     out_raster,
#     "w",
#     driver="GTiff",
#     height=height,
#     width=width,
#     count=1,
#     dtype='float32',
#     crs=landcover_with_condition.crs,
#     transform=transform,
# ) as dst:
#     dst.write(condition_raster, 1)

In [ ]:
# plt.figure(figsize=(10, 10))
# plt.imshow(landcover_with_condition_baseline, cmap='viridis')
# plt.colorbar(label='Condition Value')
# plt.title('Land Cover Condition Raster')
# plt.xlabel('Pixel Column')
# plt.ylabel('Pixel Row')
# plt.show()

In [ ]:
# # Save the 10 m raster
# out_raster = output_dir / "landcover_condition.tif"
# with rasterio.open(
#     out_raster,
#     "w",
#     driver="GTiff",
#     height=height,
#     width=width,
#     count=1,
#     dtype='float32',
#     crs=landcover_with_condition.crs,
#     transform=transform,
# ) as dst:
#     dst.write(condition_raster, 1)
# print(f"10 m resolution raster saved at: {out_raster}")

# #

In [ ]:
# def open_raster_as_array(path):
#     """
#     opens a tiff raster file as a numpy array
#     input:
#         path: path to the tiff raster (format: ...)
#     """
#     raster_path = path
#     raster = tifffile.imread(raster_path)
#     array = np.array(raster)
#     return array

In [ ]:
# # Set new resolution
# new_resolution = 100
# new_width = int((maxx - minx) / new_resolution)
# new_height = int((maxy - miny) / new_resolution)
# new_transform = from_origin(minx, maxy, new_resolution, new_resolution)



In [ ]:
# # Create an empty array for the resampled raster data
# resampled_raster = np.empty(shape=(new_height, new_width), dtype='float32')



In [ ]:
# # Resample using average method (you can change the resampling method if needed)
# reproject(
#     source=condition_raster,
#     destination=resampled_raster,
#     src_transform=transform,
#     src_crs=landcover_with_condition.crs,
#     dst_transform=new_transform,
#     dst_crs=landcover_with_condition.crs,
#     resampling=Resampling.average
# )


In [ ]:
# # Save the resampled (100 m) raster
# resampled_out_raster = output_dir / "landcover_condition_resampled.tif"
# with rasterio.open(
#     resampled_out_raster,
#     "w",
#     driver="GTiff",
#     height=new_height,
#     width=new_width,
#     count=1,
#     dtype='float32',
#     crs=landcover_with_condition.crs,
#     transform=new_transform,
# ) as dst:
#     dst.write(resampled_raster, 1)
# print(f"100 m resolution raster saved at: {resampled_out_raster}")

In [ ]:
# condition_layer = open_raster_as_array(output_dir / "landcover_condition_resampled.tif")

In [ ]:
# land_array = np.zeros_like(condition_layer)
# land_array[np.where(condition_layer != 0)] = 1

In [ ]:
# n_processes = os.cpu_count()

In [ ]:
# lambda_parameter = 5

In [ ]:
# generations_mode = "one_generation"
# number_of_species_generations = 1

In [ ]:
# connectivity_value = connectivity.landscape_connectivity(condition_layer, 
#                                                          n_processes, land_array, lambda_parameter, generations_mode, 
#                                                          number_of_species_generations)

# connectivity_value

In [ ]:
# catchment_raster = open_raster_as_array(resampled_catchment_file)
# print("Unique catchment IDs in raster:", np.unique(catchment_raster))
# baseline_condition_raster = open_raster_as_array(baseline_resampled_file)

# print(np.shape(catchment_raster))
# print(np.shape(baseline_condition_raster))

# for catchment_number in range(1, (number_catchments + 1)):
#     catchment_temp = np.zeros_like(baseline_condition_raster)
#     catchment_temp[np.where(catchment_raster == catchment_number)] = 1
#     pixel_count = np.count_nonzero(catchment_temp)
#     print(f"Catchment {catchment_number} - non-zero pixel count: {pixel_count}")

# afforested_condition_raster = open_raster_as_array(afforestable_resampled_file)

# # Setup parameters for connectivity analysis
# n_processes = os.cpu_count()
# lambda_parameter = 5
# generations_mode = "one_generation"
# number_of_species_generations = 1

# for catchment_number in range(1,(number_catchments+1)):
#     catchment_temp = np.zeros_like(baseline_condition_raster)
#     catchment_temp[np.where(catchment_raster == catchment_number)] = 1
             
#     # Compute connectivity (using your connectivity module)
#     baseline_connectivity_value = connectivity.landscape_connectivity(
#         baseline_condition_raster, 
#         n_processes, 
#         catchment_temp, 
#         lambda_parameter, 
#         generations_mode, 
#         number_of_species_generations
#     )
#     afforested_connectivity_value = connectivity.landscape_connectivity(
#         afforested_condition_raster, 
#         n_processes, 
#         catchment_temp, 
#         lambda_parameter, 
#         generations_mode, 
#         number_of_species_generations
#     )

#     perc_difference = (((afforested_connectivity_value - baseline_connectivity_value)/baseline_connectivity_value)*100) 
#     print("connectivity catchment", catchment_number, "baseline: ", baseline_connectivity_value, "afforested: ", afforested_connectivity_value, "in %: ", perc_difference) 
#     hydrobasins.loc[hydrobasins["new_id"]==catchment_number, ["baseline_catchment_connectivity"]] = baseline_connectivity_value
#     hydrobasins.loc[hydrobasins["new_id"]==catchment_number, ["afforested_catchment_connectivity"]] = afforested_connectivity_value
#     hydrobasins.loc[hydrobasins["new_id"]==catchment_number, ["percentage_difference_catchment_connectivity"]] = perc_difference 



